# IEE575 Lab 7: Bayesian Optimization
**Arizona State University — Applied Stochastic Operations Research Models**

---

Previously, you implemented a Gaussian Process regressor from scratch and saw how the posterior
updates when new observations arrive. Today we put that surrogate to work: we use it to *decide
where to evaluate an expensive function next*.

**By the end of this lab you will be able to:**
1. Explain the three components of Bayesian Optimization (surrogate, acquisition, loop)
2. Derive and implement Expected Improvement (EI) from the conditioning formula
3. Run a full BO loop on the 2D Himmelblau function and interpret each iteration
4. Diagnose two classes of BO failure: bad kernel choice and high-dimensional scaling

**Rules for today:**
- Every `???` block is a gap you must fill. You may not proceed to the next section until the current one runs correctly.
- Written reflection questions (marked **Q:**) carry marks — answer in the cell provided.
- You may use AI to *check your understanding* after you have written an answer. You may not use it to generate answers before you attempt them.


## 0. Setup

Run this cell first. Do not modify it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, Matern
from scipy.stats import norm
import time

rng = np.random.default_rng(42)   # single RNG — all randomness flows through here

BOUNDS = [-5., 5.]
GRID_N = 80
_x     = np.linspace(BOUNDS[0], BOUNDS[1], GRID_N)
X1, X2 = np.meshgrid(_x, _x)
X_grid  = np.vstack([X1.ravel(), X2.ravel()]).T   # (6400, 2)


---
## Section 1 — The Objective Function

We will optimize the **Himmelblau function**:

$$f(x, y) = (x^2 + y - 11)^2 + (x + y^2 - 7)^2$$

It has **four global minima** all with $f = 0$, located at:

| $(x^*, y^*)$ |
|---|
| $(3.0,\ 2.0)$ |
| $(-2.805,\ 3.131)$ |
| $(-3.779,\ -3.283)$ |
| $(3.584,\ -1.848)$ |

We treat $f$ as a **black box**: we can evaluate it at any point, but we cannot see its gradient and each evaluation is expensive.

---


### Task 1.1 — Implement the Himmelblau function

Implement `himmelblau(X)` where `X` is an array of shape `(n, 2)`.

**PREDICT** — before writing any code: sketch the shape of this function in your head.
How many local minima does a function with *two* squared terms tend to have?

> *Your prediction:*

**Hint:** operate directly on columns `X[:, 0]` and `X[:, 1]` — no loop needed.


In [ ]:
def himmelblau(X):
    """
    Evaluate the Himmelblau function on a batch of 2D points.

    Parameters
    ----------
    X : np.ndarray, shape (n, 2)
        Each row is one (x, y) point.

    Returns
    -------
    np.ndarray, shape (n,)
        Function values.
    """
    x = X[:, 0]
    y = X[:, 1]
    return ???   # (x^2 + y - 11)^2 + (x + y^2 - 7)^2


# Quick sanity check — the four known minima should all give ≈ 0
minima = np.array([[3.0, 2.0],
                   [-2.805118, 3.131312],
                   [-3.779310, -3.283186],
                   [3.584428, -1.848126]])
vals = himmelblau(minima)
print("Values at known minima:", np.round(vals, 4))
assert np.all(vals < 1e-3), "Check your formula — at least one minimum is wrong"


### Task 1.2 — Visualize the landscape

The cell below is partially complete. Fill in the two `???` lines, then run it.

**PREDICT** — where will the four minima appear on the contour plot?
Mark them approximately on a rough sketch before running.

> *Your sketch / description:*


In [ ]:
Z_true = himmelblau(X_grid).reshape(GRID_N, GRID_N)

fig, ax = plt.subplots(figsize=(6, 5))
cf = ax.contourf(X1, X2, ???, levels=60, cmap='viridis')   # what surface are we plotting?
plt.colorbar(cf, ax=ax, label=???                           )   # what does the color represent?
ax.set_title("Himmelblau: True Function")
ax.set_xlabel("x")
ax.set_ylabel("y")

# Mark the known minima
ax.scatter(minima[:, 0], minima[:, 1], c='red', s=80, zorder=5, label='True minima')
ax.legend()
plt.tight_layout()
plt.show()


**Q1:** Describe what you see. Why does the function have a complicated landscape with multiple basins?
What would go wrong if you tried gradient descent from a random starting point?

> *Your answer:*


---
## Section 2 — The Acquisition Function: Expected Improvement

The surrogate gives us a posterior $\mu(x)$ and $\sigma(x)$ everywhere.
We need a rule to decide *where to query next*.

### Derivation

Let $f^* = \max_{i} y_i$ be the best value observed so far (we negate $f$ so the GP maximizes).

Define the **improvement** at a candidate point $x$ as:

$$I(x) = \max(f(x) - f^* - \xi,\ 0)$$

where $\xi > 0$ is a small exploration bonus.

Since $f(x)$ is uncertain, we model it as $f(x) \sim \mathcal{N}(\mu(x), \sigma^2(x))$.

Taking the expectation over this distribution:

$$\text{EI}(x) = \mathbb{E}[I(x)] = (\mu(x) - f^* - \xi)\,\Phi(Z) + \sigma(x)\,\phi(Z)$$

where:
- $Z = \dfrac{\mu(x) - f^* - \xi}{\sigma(x)}$
- $\Phi$ is the standard normal CDF (`norm.cdf`)
- $\phi$ is the standard normal PDF (`norm.pdf`)
- EI is defined as 0 wherever $\sigma(x) \approx 0$ (already observed, no uncertainty)

---

### Task 2.1 — Implement EI

Translate each step of the derivation above into code. The formula is given; your job is to map it
to the right variables.


In [ ]:
def expected_improvement(X_cand, X_sample, Y_sample, gp, xi=0.01):
    """
    Compute Expected Improvement at candidate points X_cand.

    Parameters
    ----------
    X_cand   : np.ndarray (m, d)  — candidate points to evaluate EI at
    X_sample : np.ndarray (n, d)  — observed input locations
    Y_sample : np.ndarray (n,)    — observed (negated) function values
    gp       : fitted GaussianProcessRegressor
    xi       : float              — exploration bonus (default 0.01)

    Returns
    -------
    np.ndarray (m,) — EI values at each candidate point
    """
    # Step 1: Get posterior mean and std at candidate points
    mu, sigma = gp.predict(???, return_std=True)

    # Step 2: Find the best observed value in Y_sample
    #         (Y_sample stores -f values, so best = maximum)
    f_star = np.max(gp.predict(X_sample))

    # Step 3: Guard against numerical zero sigma
    sigma = np.maximum(sigma, 1e-9)

    # Step 4: Compute Z  =  (mu - f_star - xi) / sigma
    Z = ???

    # Step 5: Compute EI  =  (mu - f_star - xi) * Phi(Z)  +  sigma * phi(Z)
    ei = ???

    # Step 6: Zero out EI wherever sigma was essentially 0
    ei[sigma < 1e-9] = 0.0

    return ei


**Q2:** EI has two terms: $(\mu - f^* - \xi)\Phi(Z)$ and $\sigma \phi(Z)$.
What does each term encourage the optimizer to do? Which term dominates early in the search?
Which dominates late?

> *Your answer:*

**Q3:** What is the role of $\xi$? What happens if $\xi = 0$? What if $\xi$ is very large?

> *Your answer:*


---
## Section 3 — The Bayesian Optimization Loop

The BO loop has three steps per iteration:

1. **Fit** the GP surrogate to all observations so far
2. **Maximize** EI over the search space to find the next query point
3. **Evaluate** the true function at that point and append to the dataset

---

### Task 3.1 — Initial data

We start with a small random design. Read the code carefully before running.


In [ ]:
n_init = 20

# We draw the initial sample once and keep it as the reference copy.
# Both the RBF run (Section 3) and the Matérn-1/2 run (Section 4A) will start
# from this exact same set of points — the only variable between the two runs
# is the kernel choice.
X_sample_init = rng.uniform(BOUNDS[0], BOUNDS[1], (n_init, 2))
Y_sample_init = himmelblau(X_sample_init)

# Working copies for Section 3 — do not overwrite X_sample_init / Y_sample_init
X_sample = X_sample_init.copy()
Y_sample = Y_sample_init.copy()

# convergence_init tracks the best value seen across the shared init points,
# one entry per point. Both the RBF and Matérn curves will start from this
# prefix so their convergence plots are directly comparable.
convergence_init = [float(Y_sample_init[:k+1].min()) for k in range(n_init)]

print(f"Initial dataset: {n_init} points")
print(f"Best initial value: {Y_sample.min():.3f}  (true minimum = 0)")
print(f"X_sample shape: {X_sample.shape}")


### Task 3.2 — GP model

We use sklearn's `GaussianProcessRegressor` with a `ConstantKernel * RBF` kernel.
The kernel is **already set up for you** — your task is to understand it.


In [ ]:
kernel = C(1.0, (1e-3, 1e3)) * RBF(length_scale=1.0)

gp = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=5,
    normalize_y=True
)

print("Kernel before fitting:", gp.kernel)


**Q4:** Why do we pass `-1 * Y_sample` to `gp.fit()` instead of `Y_sample` directly?
(Hint: look at how EI is defined — it assumes we are *maximizing*.)

> *Your answer:*


### Task 3.3 — The BO loop

The loop structure is given. Fill in the four `???` blocks.

**PREDICT** — before running: given 20 random start points on Himmelblau, which of the four
minima do you expect BO to find first? Why?

> *Your prediction:*


In [ ]:
n_iter = 15
# Start convergence from the shared init prefix so it is directly comparable
# with the Matérn-1/2 curve in Section 4A.
convergence = convergence_init.copy()

for i in range(n_iter):

    # --- Step 1: Fit surrogate ---
    # We negate Y because the GP / EI maximizes, but Himmelblau is a minimization problem
    gp.fit(X_sample, ??? * Y_sample)

    # --- Step 2: GP predictions over the full grid ---
    mu, sigma = gp.predict(X_grid, return_std=True)
    mu    = mu.reshape(GRID_N, GRID_N)
    sigma = sigma.reshape(GRID_N, GRID_N)

    # --- Step 3: Compute EI and find next query point ---
    ei     = expected_improvement(X_grid, X_sample, Y_sample, gp)
    ei_map = ei.reshape(GRID_N, GRID_N)
    x_next = X_grid[np.argmax(???)]           # the candidate with highest EI

    # --- Step 4: Evaluate true function and update dataset ---
    y_next    = himmelblau(x_next.reshape(1, -1))
    X_sample  = np.vstack([X_sample, x_next])
    Y_sample  = np.append(Y_sample, y_next)

    best_so_far = Y_sample.min()
    convergence.append(best_so_far)

    # --- Plotting (every 5 iterations) ---
    if (i + 1) % 5 == 0:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        # Panel 1: True function + all observations
        axes[0].contourf(X1, X2, Z_true, levels=50, cmap='viridis')
        axes[0].scatter(X_sample[:-1, 0], X_sample[:-1, 1], c='white', s=15, label='observed')
        axes[0].scatter(x_next[0], x_next[1], c='red', s=80, marker='*', label='just queried')
        axes[0].set_title(f"True Function — Iter {i+1}")
        axes[0].legend(fontsize=8)

        # Panel 2: GP mean — fill in what to show and add a colorbar label
        axes[1].contourf(X1, X2, ???, levels=50, cmap='plasma')   # GP mean or sigma?
        axes[1].scatter(X_sample[:, 0], X_sample[:, 1], c='white', s=15)
        axes[1].set_title(???)                                      # descriptive title

        # Panel 3: Acquisition function — fill in what to show
        axes[2].contourf(X1, X2, ???, levels=50, cmap='inferno')
        axes[2].scatter(X_sample[:, 0], X_sample[:, 1], c='white', s=15)
        axes[2].scatter(x_next[0], x_next[1], c='red', s=80, marker='*')
        axes[2].set_title(???)                                      # descriptive title

        plt.suptitle(f"BO Iteration {i+1}  |  Best f = {best_so_far:.4f}", fontsize=13)
        plt.tight_layout()
        plt.show()

print(f"\nFinal best value: {Y_sample.min():.6f}")
print(f"Final best location: {X_sample[np.argmin(Y_sample)]}")


### Task 3.4 — Convergence plot

Plot `convergence` (the list of best values per iteration) vs. iteration number.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

# YOUR CODE HERE: plot convergence vs iteration
# Label axes clearly. Add a horizontal dashed line at y=0 (the true optimum).

plt.tight_layout()
plt.show()


**Q5:** Describe the convergence curve. Is improvement monotone? Why must it be monotone
(or: why can't the best value get *worse* between iterations)?

> *Your answer:*

**Q6:** Look at the iteration plots. Does BO explore the whole domain or does it exploit
one basin early? How does the EI map change as more points are observed?

> *Your answer:*


---
## Section 4 — When BO Fails: The Wrong Surrogate

The GP surrogate is the engine of BO. If the surrogate is wrong, the acquisition function
optimizes a *fiction* — and BO converges to the wrong place with full confidence.

We now examine two failure modes systematically.

---

### 4A — Bad Kernel Choice: Matérn-1/2

The RBF kernel assumes the function is *infinitely differentiable* — infinitely smooth.
The Matérn-1/2 kernel assumes the function is only **continuous but not differentiable anywhere** —
it models rough, jagged functions.

Himmelblau is a smooth polynomial. Watch what happens when we lie to the GP about this.


### Task 4.1 — Run BO with Matérn-1/2 kernel

The loop below is identical to Section 3 except for the kernel. Run it and observe.

**PREDICT** — before running: how will the GP mean surface look different?
Where do you expect BO to waste evaluations?

> *Your prediction:*


In [ ]:
# Bad kernel: Matern-1/2 is appropriate for very rough (non-differentiable) functions.
# Himmelblau is a smooth polynomial — this is a deliberate mismatch.
kernel_bad = C(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, nu=0.5)

gp_bad = GaussianProcessRegressor(
    kernel=kernel_bad,
    n_restarts_optimizer=5,
    normalize_y=True
)

# Start from the SAME initial sample as Section 3.
# Only the kernel changes — nothing else.
X_sample_bad = X_sample_init.copy()
Y_sample_bad = Y_sample_init.copy()

# Start from the same init prefix as the RBF run so both curves are comparable.
convergence_bad = convergence_init.copy()

for i in range(n_iter):
    gp_bad.fit(X_sample_bad, -1 * Y_sample_bad)

    mu_bad, sigma_bad = gp_bad.predict(X_grid, return_std=True)
    mu_bad    = mu_bad.reshape(GRID_N, GRID_N)
    sigma_bad = sigma_bad.reshape(GRID_N, GRID_N)

    ei_bad = expected_improvement(X_grid, X_sample_bad, Y_sample_bad, gp_bad)
    x_next_bad = X_grid[np.argmax(ei_bad)]
    y_next_bad = himmelblau(x_next_bad.reshape(1, -1))

    X_sample_bad = np.vstack([X_sample_bad, x_next_bad])
    Y_sample_bad = np.append(Y_sample_bad, y_next_bad)
    convergence_bad.append(Y_sample_bad.min())

    if (i + 1) % 5 == 0:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        axes[0].contourf(X1, X2, Z_true, levels=50, cmap='viridis')
        axes[0].scatter(X_sample_bad[:-1, 0], X_sample_bad[:-1, 1], c='white', s=15)
        axes[0].scatter(x_next_bad[0], x_next_bad[1], c='red', s=80, marker='*')
        axes[0].set_title(f"True Function — Iter {i+1} [Matérn-1/2]")

        axes[1].contourf(X1, X2, mu_bad, levels=50, cmap='plasma')
        axes[1].scatter(X_sample_bad[:, 0], X_sample_bad[:, 1], c='white', s=15)
        axes[1].set_title("GP Mean (Matérn-1/2)")

        axes[2].contourf(X1, X2, ei_bad.reshape(GRID_N, GRID_N), levels=50, cmap='inferno')
        axes[2].scatter(X_sample_bad[:, 0], X_sample_bad[:, 1], c='white', s=15)
        axes[2].scatter(x_next_bad[0], x_next_bad[1], c='red', s=80, marker='*')
        axes[2].set_title("EI (Matérn-1/2)")

        plt.suptitle(f"Matérn-1/2 BO — Iter {i+1}  |  Best f = {Y_sample_bad.min():.4f}", fontsize=13)
        plt.tight_layout()
        plt.show()


### Task 4.2 — Compare convergence curves

Plot both convergence curves (RBF and Matérn-1/2) on the same axes.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

# YOUR CODE HERE
# Both convergence lists now include the shared init prefix (first n_init entries)
# followed by the BO iterations. Plot them on the same axes.
#
# Hints:
#   x-axis: range(1, n_init + n_iter + 1)
#   shade the shared init with ax.axvspan(1, n_init, alpha=0.10, color='grey')
#   add ax.axvline(n_init, color='grey', lw=1, linestyle=':')
#   label axes and add a legend
#   add a dashed line at y=0 (the true optimum)

plt.tight_layout()
plt.show()


**Q7:** Describe the visual difference between the GP mean surfaces produced by RBF
and Matérn-1/2. Why does the Matérn-1/2 surface look the way it does on a smooth function?

> *Your answer:*

**Q8:** Does the wrong kernel prevent BO from converging, or does it just slow it down?
What would happen if the true function *were* rough (e.g., had discontinuities)?
Would RBF or Matérn-1/2 be the bad choice then?

> *Your answer:*


---
### 4B — High-Dimensional Scaling: When the GP Breaks Down

The RBF kernel in $d$ dimensions is:

$$k(\mathbf{x}, \mathbf{x'}) = \exp\left(-\frac{\|\mathbf{x} - \mathbf{x'}\|^2}{2\ell^2}\right)$$

Fitting a GP requires solving a linear system with the $n \times n$ kernel matrix $K(X, X)$.
This costs $\mathcal{O}(n^3)$ in time and $\mathcal{O}(n^2)$ in memory.

More critically, the **volume of the search space** grows exponentially with $d$:
a grid with 10 points per axis requires $10^d$ evaluations to cover the space.
This is the **curse of dimensionality**.

In the experiment below we hold the number of observations fixed and increase $d$,
measuring how long each GP fit takes and how well BO can explore the space.


### Task 4.3 — Timing experiment: GP fit cost vs. dimension

We extend Himmelblau to higher dimensions by adding dummy dimensions with no effect on $f$:

$$f_{\text{ext}}(\mathbf{x}) = (x_1^2 + x_2 - 11)^2 + (x_1 + x_2^2 - 7)^2$$

The remaining $d - 2$ dimensions are ignored by the function but the GP doesn't know that
— it must learn their irrelevance from data.

Fill in the `???` in the timing loop.

**PREDICT** — how do you expect fit time to scale with $d$? Linear? Quadratic? Worse?

> *Your prediction:*


In [ ]:
def himmelblau_extended(X):
    """Himmelblau on first 2 dims; remaining dims are irrelevant."""
    return himmelblau(X[:, :2])


dims      = [2, 5, 10, 20, 50]
n_obs     = 100    # fixed number of observations
fit_times = []

for d in dims:
    # Generate random observations in d dimensions
    X_d = rng.uniform(-5, 5, (n_obs, d))
    Y_d = himmelblau_extended(X_d)

    # Build a d-dimensional GP
    kernel_d = C(1.0) * RBF(length_scale=np.ones(d))   # one length-scale per dimension
    gp_d = GaussianProcessRegressor(kernel=kernel_d, n_restarts_optimizer=0, normalize_y=True)

    # --- Time the fit ---
    t_start = time.time()
    gp_d.fit(???, ??? * Y_d)    # fit on X_d with negated Y
    t_end = time.time()

    fit_times.append(t_end - t_start)
    print(f"d={d:3d}  |  fit time: {fit_times[-1]:.3f}s")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

# YOUR CODE HERE
# Plot dims vs fit_times. Use a log scale on the y-axis (ax.set_yscale('log')).
# Label axes. Add a title explaining what this plot shows.

plt.tight_layout()
plt.show()


### Task 4.4 — Coverage collapse

In low dimensions, $n=100$ observations can cover the space reasonably well.
In high dimensions, the same $n$ observations are a vanishing fraction of the volume.

The cell below computes the **average nearest-neighbor distance** as a proxy for coverage.
A large average distance means the surrogate is interpolating across huge gaps.


In [ ]:
from sklearn.metrics import pairwise_distances

coverage = []

for d in dims:
    X_d = rng.uniform(-5, 5, (n_obs, d))

    # Pairwise distances between all observations
    D = pairwise_distances(X_d)
    np.fill_diagonal(D, np.inf)   # ignore self-distance

    # Average distance to nearest neighbor
    avg_nn_dist = np.mean(np.min(D, axis=1))
    coverage.append(avg_nn_dist)
    print(f"d={d:3d}  |  avg nearest-neighbor distance: {avg_nn_dist:.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

# YOUR CODE HERE
# Plot dims vs coverage.
# What pattern do you see? What does this mean for the surrogate's reliability?

plt.tight_layout()
plt.show()


**Q9:** Interpret the nearest-neighbor distance plot. What does it tell you about
the GP surrogate's reliability as $d$ grows? How does this interact with the timing
result from Task 4.3 — i.e., the GP gets *slower* and *less reliable* simultaneously?

> *Your answer:*

**Q10:** In practice, real engineering and ML problems often have 50–1000 tunable parameters.
Based on what you observed, why is vanilla BO not viable at that scale?

> *Your answer:*


---
## Section 5 — Beyond Vanilla BO (Discussion)

The two failure modes you just witnessed motivate a family of algorithms designed to
extend BO to settings where the vanilla GP breaks down.

| Algorithm | Core Idea | Addresses |
|---|---|---|
| **TuRBO** (Trust Region BO) | Restricts the search to a local trust region around the current best; shrinks/expands based on progress | High dimensions: focuses budget where it matters |
| **SAASBO** (Sparse Axis-Aligned Subspace BO) | Places a sparsity prior over length-scales; learns which dimensions actually matter | High dimensions with many irrelevant inputs |
| **BAXUS** | Adaptively embeds the problem in a low-dimensional subspace | Extreme high-d with low effective dimensionality |
| **Deep Kernel Learning** | Replaces the fixed kernel with a neural network feature map | Complex structure that fixed kernels can't capture |

You are not expected to implement these. The key insight is:

> **Every extension to vanilla BO is a response to a specific failure mode of the GP surrogate.**
> Understanding *why* BO fails tells you *which* algorithm to reach for.

---

**Q11 (Open-ended):** Suppose you are tuning a neural network with 100 hyperparameters,
but you suspect only 5–10 of them actually matter. Which algorithm from the table above
would you try first and why? What information would you use to check your assumption?

> *Your answer:*
